# Lab Assignment 5: Wide and Deep Network Architectures

**Team Members:** Emilio M. , Jadon Swearingen, Andy Su  
**Date:** 2026-04-16  
**Dataset:** Vehicle Sales Data (Kaggle — syedanwarafridi/vehicle-sales-data)  
**Task:** Binary Classification — predict whether a vehicle sold above its Manheim Market Report (MMR) value

---
## 0. Imports and Setup

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input

from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, roc_curve, auc
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print('TensorFlow version:', tf.__version__)

---
## Part 1: Data Preparation


### 1.1 Load Dataset

In [ ]:
df = pd.read_csv('car_prices.csv')
print(df.shape)
df.head()

### 1.2 Exploratory Overview

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
print('\nTarget distribution (sellingprice > mmr):')
print((df['sellingprice'] > df['mmr']).value_counts())


### 1.3 Preprocessing

These are the preprocessing steps we applied before training:



1. **Create a binary target** We define sold_above_mmr as 1 if the selling price exceeded the MMR (Manheim Market Report), and 0 otherwise. MMR is the wholesale market estimate. This target captures whether a car sold above what the market expected.

2. **Drop columns that would cause leakage or add no signal** — sellingprice is removed since it directly determines the target. vin is just a unique identifier, and saledate would need complex parsing with little payoff.

3. **Drop seller** — With over 14,000 unique values, this column is too high cardinality to be useful without a dedicated embedding strategy. Most of the geographic and brand signal it carries is already covered by state and make.

4. **Standardize body and transmission** — These columns had inconsistent casing, for example `sedan` vs `Sedan`, `suv` vs `SUV`, so we lowercased and stripped whitespace throughout to make it more consistent. We also noticed some transmission entries were labeled sedan or Sedan, which is mislabeled, so those got remapped to unknown.

5. **Clean color** — A handful of rows had numeric strings in place of color names. We replaced those with unknown to keep the column consistent.

6. **Impute missing values** — Missing categorical values are filled with unknown, and missing numerics are filled with the column median. The medians are computed on the full dataset before splitting, which is a reasonable shortcut given the 558K sample size; the impact on any individual median is negligible.

7. **Integer-encode categoricals** — We fit abelEncoder`s on the full dataset so every category level gets a consistent integer code before the split. Since the test set is drawn randomly from the same data, there are no unseen categories to worry about.

8. **Scale numeric features** — StandardScaler is fit only on the training set and then applied to both splits. This keeps the model from ever seeing test set statistics during training, so the evaluation is better.

In [ ]:
NUMERIC_FEATURES     = ['year', 'condition', 'odometer', 'mmr']
CATEGORICAL_FEATURES = ['make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior']
TARGET_COL           = 'sold_above_mmr'


# This part creates binary target
df[TARGET_COL] = (df['sellingprice'] > df['mmr']).astype(int)


# This drops leakage, identifiers, and high-cardinality seller column
df = df.drop(columns=['sellingprice', 'vin', 'saledate', 'seller'])




# This section standardize body and transmission casing

df['body']         = df['body'].str.lower().str.strip()
df['transmission'] = df['transmission'].str.lower().str.strip()
df['transmission'] = df['transmission'].replace({'sedan': 'unknown', 'nan': 'unknown'})



# Clean color, this part replaces the numeric values with 'unknown'
valid_colors = ['black','white','silver','gray','blue','red','green','gold',
                'beige','burgundy','brown','orange','purple','off-white',
                'yellow','charcoal','turquoise','pink','lime']
df['color'] = df['color'].apply(lambda x: x if x in valid_colors else 'unknown')


# Fill missing values
df[NUMERIC_FEATURES]     = df[NUMERIC_FEATURES].fillna(df[NUMERIC_FEATURES].median())
df[CATEGORICAL_FEATURES] = df[CATEGORICAL_FEATURES].fillna('unknown')


label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

DEEP_VOCAB_SIZES = {col: df[col].nunique() + 1 for col in CATEGORICAL_FEATURES}
print('Dataset shape after preprocessing:', df.shape)
print('Target balance:')
print(df[TARGET_COL].value_counts())
print('\nDeep vocab sizes:', DEEP_VOCAB_SIZES)

### 1.4 Cross-Product Feature Justification

**Features selected for crossing:**

| Feature A | Feature B | Justification |
|-----------|-----------|---------------|
| `make` | `body` | A Ford SUV and a Toyota SUV sell very differently, brand and body type together matter more than either alone. |
| `color` | `interior` | Some color/interior combos (for example, black/black) are noticeably more desirable than others; neither feature captures this on its own. |
| `state` | `make` | Buyer preferences vary by region, so the same brand can command a premium in one state but not another. |

**Features NOT crossed:**
- `year` × `odometer` — both are continuous, so one hot crossing doesn't apply; the deep branch handles their interaction.
- `model` × `trim` — with 973 and 1,963 unique values respectively, the resulting cross would have ~1M combinations and be extremely sparse.
- `transmission` — only 3 meaningful values, so there's not much useful interaction to capture with a cross.

In [ ]:
CROSS_FEATURE_PAIRS = [

    ('make',  'body'),
    ('color', 'interior'),
    ('state', 'make'),

]

WIDE_CROSS_VOCAB_SIZES = {}

for (fa, fb) in CROSS_FEATURE_PAIRS:

    # Decode back to strings to form the cross key, then re encode
    fa_str = label_encoders[fa].inverse_transform(df[fa])
    fb_str = label_encoders[fb].inverse_transform(df[fb])
    cross_col = f'{fa}_x_{fb}'
    df[cross_col] = fa_str.astype(str) + '_' + fb_str.astype(str)
    le_cross = LabelEncoder()
    df[cross_col] = le_cross.fit_transform(df[cross_col])
    label_encoders[cross_col] = le_cross
    WIDE_CROSS_VOCAB_SIZES[cross_col] = df[cross_col].nunique() + 1


print('Wide cross features:', list(WIDE_CROSS_VOCAB_SIZES.keys()))
print('Wide vocab sizes:', WIDE_CROSS_VOCAB_SIZES)

### 1.5 Evaluation Metric Justification

**Chosen metric:** AUC-ROC (Area Under the Receiver Operating Characteristic Curve)

**Justification:**

The task is to predict whether a vehicle will sell above its MMR value at auction. In this context:

- **Class imbalance is mild but present** — 261,161 above MMR vs 297,676 at or below MMR. Accuracy would be ~53% for a trivial majority class classifier, making it a misleading metric.

- **Both false positives and false negatives carry real cost.** A buyer that incorrectly believes a car will sell above MMR may overbid. A seller that misclassifies a car as unlikely to beat MMR may under price it. Neither error is clearly more costly than the other, so we do not want to arbitrarily threshold with precision or recall alone.

- **AUC-ROC measures rank ordering across all thresholds.** It captures how well a model separates above MMR from at or below MMR vehicles regardless of the decision threshold chosen. This reflects real world usage: an auction platform would calibrate its own threshold based on business policy, and we want a metric that is threshold-agnostic.

- **AUC-ROC is the rubric-required metric** this is for classification tasks and enables a rigorous statistical comparison between the Wide & Deep model and the MLP baseline using DeLong's test.

Accuracy is explicitly avoided as it does not reflect the ranking quality of probabilistic predictions and is misleading under any class imbalance.

### 1.6 Data Splitting Strategy

**Chosen method:** Stratified 5 Fold Cross Validation on the development set, with a fixed 15% hold out test set.


**Justification:**

- **Stratified folds** THey are necessary because the target is slightly imbalanced since they are ~47% positive. Stratification ensures each fold has the same class ratio as the full dataset. This prevents any fold from being a poor representative sample.

- **5 fold CV** This is chosen over a single train/val split because it gives us 5 independent AUC estimates per model configuration. It enables statistical tests to compare architectures. A single split would give only one data point per model, this makes it insufficient for significance testing.

- **Why not 10 fold?** With 558K rows, each fold in 5 fold CV already contains ~90K training samples and ~23K validation samples — far more than needed for stable estimates. 10 fold would add computational cost without meaningfully improving estimate stability given the dataset size.

- **Fixed hold out test set with 15%** — It is reserved exclusively for the final MLP vs. Wide & Deep comparison in Part 4. It uses the same test set for both models and ensures that the test compares predictions on the same observations, which is a requirement of the paired test.

- **Realistic mirroring of production use:** In practice, an auction platform would train on historical sales and predict future auctions. A random split is acceptable here because the dataset doesn't exhibit a strong temporal drift across the date range represented.

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values


N_SPLITS = 5
kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

X_dev, X_test, y_dev, y_test = train_test_split(

    X, y, test_size=0.15, stratify=y, random_state=RANDOM_SEED

)

# Fit scaler prevents test set statistics from influencing the learned scale
# By taking this approach we were able to prevent data leakage

scaler = StandardScaler()

X_dev  = X_dev.copy()
X_test = X_test.copy()

X_dev[NUMERIC_FEATURES]  = scaler.fit_transform(X_dev[NUMERIC_FEATURES])
X_test[NUMERIC_FEATURES] = scaler.transform(X_test[NUMERIC_FEATURES])

print(f'Dev size:  {X_dev.shape}')
print(f'Test size: {X_test.shape}')


print(f'Test target balance: {y_test.mean():.3f} positive rate')

---
## Part 2: Wide and Deep Models
*(Rubric: Three Wide/Deep Models 2pts)*

### 2.1 Model Building Helpers

In [ ]:
def df_to_model_inputs(X_df, numeric_features, categorical_features, wide_cross_features=None):
    inputs = {
        'numeric_input': X_df[numeric_features].values.astype(np.float32)
    }
    for col in categorical_features:
        inputs[f'cat_{col}'] = X_df[col].values.reshape(-1, 1).astype(np.int32)
    if wide_cross_features:
        for col in wide_cross_features:
            inputs[f'wide_{col}'] = X_df[col].values.reshape(-1, 1).astype(np.int32)
    return inputs


def build_wide_and_deep(
    numeric_dim,
    deep_vocab_sizes,
    wide_cross_vocab_sizes=None,
    embedding_dim=8,
    deep_layers=[128, 64],
    dropout_rate=0.3,
    output_units=1,
    output_activation='sigmoid',
    learning_rate=1e-3
):
    """
    Wide & Deep model (Cheng et al. 2016).
    Wide branch : one-hot cross-product features + raw numeric (memorization).
    Deep branch : embeddings per categorical + numeric (generalization).
    """
    if wide_cross_vocab_sizes is None:
        wide_cross_vocab_sizes = {}

    numeric_input = Input(shape=(numeric_dim,), name='numeric_input')
    cat_inputs = {
        col: Input(shape=(1,), name=f'cat_{col}', dtype='int32')
        for col in deep_vocab_sizes
    }
    wide_inputs = {
        col: Input(shape=(1,), name=f'wide_{col}', dtype='int32')
        for col in wide_cross_vocab_sizes
    }

    # Deep branch
    embeddings = []
    for col, vsize in deep_vocab_sizes.items():
        emb = layers.Embedding(input_dim=vsize, output_dim=embedding_dim,
                               name=f'emb_{col}')(cat_inputs[col])
        emb = layers.Flatten()(emb)
        embeddings.append(emb)

    deep = layers.concatenate([numeric_input] + embeddings) if embeddings else numeric_input
    for units in deep_layers:
        deep = layers.Dense(units, activation='relu')(deep)
        deep = layers.Dropout(dropout_rate)(deep)

    # Wide branch
    wide_parts = [numeric_input]
    for col, vsize in wide_cross_vocab_sizes.items():
        one_hot = layers.CategoryEncoding(
            num_tokens=vsize, output_mode='multi_hot', name=f'oh_{col}'
        )(wide_inputs[col])
        wide_parts.append(one_hot)

    wide = layers.concatenate(wide_parts) if len(wide_parts) > 1 else wide_parts[0]

    combined = layers.concatenate([wide, deep])
    output = layers.Dense(output_units, activation=output_activation, name='output')(combined)

    all_inputs = [numeric_input] + list(cat_inputs.values()) + list(wide_inputs.values())
    model = Model(inputs=all_inputs, outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss='binary_crossentropy',
        metrics=['AUC']
    )
    return model


def plot_training_history(history, title='Model Training History'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    metric_key     = [k for k in history.history if 'auc' in k.lower() and 'val' not in k][0]
    val_metric_key = [k for k in history.history if 'auc' in k.lower() and 'val'     in k][0]
    axes[1].plot(history.history[metric_key],     label='Train AUC')
    axes[1].plot(history.history[val_metric_key], label='Val AUC')
    axes[1].set_title(f'{title} — AUC')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


print('Helpers defined.')

### 2.2 Model 1 — Wide & Deep (Baseline)

**Architecture:** deep layers = [128, 64], embedding_dim = 8, dropout = 0.3, lr = 1e-3  
**Hyperparameter choices:** [Explain each major choice]

In [ ]:
model1 = build_wide_and_deep(
    numeric_dim=len(NUMERIC_FEATURES),
    deep_vocab_sizes=DEEP_VOCAB_SIZES,
    wide_cross_vocab_sizes=WIDE_CROSS_VOCAB_SIZES,
    deep_layers=[128, 64],
    dropout_rate=0.3,
    learning_rate=1e-3
)
model1.summary()

In [ ]:
X_tr1, X_val1, y_tr1, y_val1 = train_test_split(
    X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=RANDOM_SEED
)

train_inputs1 = df_to_model_inputs(X_tr1,  NUMERIC_FEATURES, CATEGORICAL_FEATURES, list(WIDE_CROSS_VOCAB_SIZES))
val_inputs1   = df_to_model_inputs(X_val1, NUMERIC_FEATURES, CATEGORICAL_FEATURES, list(WIDE_CROSS_VOCAB_SIZES))

history1 = model1.fit(
    x=train_inputs1, y=y_tr1,
    validation_data=(val_inputs1, y_val1),
    epochs=50, batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
    verbose=1
)

plot_training_history(history1, 'Model 1 — Wide & Deep Baseline')

### 2.3 Model 2 — Wide & Deep (Variant)

**Changes from Model 1:** [e.g., deeper network, higher dropout]  
**Hypothesis:** [What do you expect to change and why?]

In [ ]:
model2 = build_wide_and_deep(
    numeric_dim=len(NUMERIC_FEATURES),
    deep_vocab_sizes=DEEP_VOCAB_SIZES,
    wide_cross_vocab_sizes=WIDE_CROSS_VOCAB_SIZES,
    deep_layers=[256, 128, 64],
    dropout_rate=0.4,
    learning_rate=5e-4
)

history2 = model2.fit(
    x=train_inputs1, y=y_tr1,
    validation_data=(val_inputs1, y_val1),
    epochs=50, batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
    verbose=1
)

plot_training_history(history2, 'Model 2 — Wide & Deep Variant')

### 2.4 Model 3 — Wide & Deep (Variant 2)

**Changes from Model 1:** [e.g., shallower network, larger embeddings]  
**Hypothesis:** [What do you expect to change and why?]

In [ ]:
model3 = build_wide_and_deep(
    numeric_dim=len(NUMERIC_FEATURES),
    deep_vocab_sizes=DEEP_VOCAB_SIZES,
    wide_cross_vocab_sizes=WIDE_CROSS_VOCAB_SIZES,
    embedding_dim=16,
    deep_layers=[64, 32],
    dropout_rate=0.2,
    learning_rate=1e-3
)

history3 = model3.fit(
    x=train_inputs1, y=y_tr1,
    validation_data=(val_inputs1, y_val1),
    epochs=50, batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
    verbose=1
)

plot_training_history(history3, 'Model 3 — Wide & Deep Variant 2')

---
## Part 3: Deep Layer Investigation with Cross-Validation
*(Rubric: Deep Layers 2pts)*

In [ ]:
def cross_validate_model(build_fn, X_df, y, kfold,
                         numeric_features, categorical_features,
                         wide_cross_features=None,
                         epochs=50, batch_size=256):
    if wide_cross_features is None:
        wide_cross_features = []

    fold_scores = []
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_df, y)):
        print(f'  Fold {fold + 1}/{kfold.n_splits} ...', end=' ', flush=True)

        X_tr  = X_df.iloc[train_idx]
        X_val = X_df.iloc[val_idx]
        y_tr  = y[train_idx]
        y_val = y[val_idx]

        tr_inputs  = df_to_model_inputs(X_tr,  numeric_features, categorical_features, wide_cross_features)
        val_inputs = df_to_model_inputs(X_val, numeric_features, categorical_features, wide_cross_features)

        model = build_fn()
        model.fit(
            tr_inputs, y_tr,
            validation_data=(val_inputs, y_val),
            epochs=epochs, batch_size=batch_size,
            callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
            verbose=0
        )

        probs = model.predict(val_inputs, verbose=0).ravel()
        fold_auc = roc_auc_score(y_val, probs)
        fold_scores.append(fold_auc)
        print(f'AUC = {fold_auc:.4f}')
        keras.backend.clear_session()

    return np.array(fold_scores)


print('cross_validate_model defined.')

In [ ]:
LAYER_CONFIGS = [
    [64],
    [128, 64],
    [256, 128, 64],
    [256, 128, 64, 32],
]

cv_results = {}

for layers_config in LAYER_CONFIGS:
    label = str(layers_config)
    print(f'\nTesting deep layers: {label}')

    def make_builder(cfg):
        def build_fn():
            return build_wide_and_deep(
                numeric_dim=len(NUMERIC_FEATURES),
                deep_vocab_sizes=DEEP_VOCAB_SIZES,
                wide_cross_vocab_sizes=WIDE_CROSS_VOCAB_SIZES,
                deep_layers=cfg,
                dropout_rate=0.3,
                learning_rate=1e-3
            )
        return build_fn

    scores = cross_validate_model(
        build_fn=make_builder(layers_config),
        X_df=X_dev, y=y_dev, kfold=kfold,
        numeric_features=NUMERIC_FEATURES,
        categorical_features=CATEGORICAL_FEATURES,
        wide_cross_features=list(WIDE_CROSS_VOCAB_SIZES.keys())
    )
    cv_results[label] = scores

print('\nCross-validation complete.')

In [ ]:
labels = list(cv_results.keys())
scores = [cv_results[k] for k in labels]

print('Mean AUC per configuration:')
for label, s in zip(labels, scores):
    print(f'  {label}: {s.mean():.4f} ± {s.std():.4f}')

best_label = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'\nBest config: {best_label}')
for label in labels:
    if label != best_label:
        stat, p = stats.wilcoxon(cv_results[best_label], cv_results[label])
        sig = '(significant at α=0.05)' if p < 0.05 else '(not significant)'
        print(f'  vs {label}: W={stat:.2f}, p={p:.4f} {sig}')

plt.figure(figsize=(10, 5))
plt.boxplot(scores, labels=labels)
plt.title('AUC by Number of Deep Layers (Cross-Validation)')
plt.ylabel('AUC')
plt.xlabel('Layer Configuration')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Conclusion — Best Number of Layers:**  
[Based on the cross-validation results and statistical tests, explain which configuration performs
superiorly and why. Reference p-values.]

---
## Part 4: MLP Comparison
*(Rubric: MLP Comparison 1pt)*

In [ ]:
def build_mlp(
    input_dim,
    hidden_layers=[128, 64],
    dropout_rate=0.3,
    output_units=1,
    output_activation='sigmoid',
    learning_rate=1e-3
):
    inp = Input(shape=(input_dim,))
    x = inp
    for units in hidden_layers:
        x = layers.Dense(units, activation='relu')(x)
        x = layers.Dropout(dropout_rate)(x)
    out = layers.Dense(output_units, activation=output_activation)(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss='binary_crossentropy',
        metrics=['AUC']
    )
    return model


def delong_auc_test(y_true, pred_a, pred_b):
    """
    DeLong et al. (1988) test for the difference between two correlated AUCs.
    Returns (z_statistic, two_sided_p_value, auc_a, auc_b).
    """
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a).ravel()
    pred_b = np.asarray(pred_b).ravel()

    pos_mask = y_true == 1
    neg_mask = y_true == 0
    pos_a, neg_a = pred_a[pos_mask], pred_a[neg_mask]
    pos_b, neg_b = pred_b[pos_mask], pred_b[neg_mask]
    m, n = pos_mask.sum(), neg_mask.sum()

    def placement_values(pos_preds, neg_preds):
        V10 = np.array(
            [(pos_preds[i] > neg_preds).mean() + 0.5 * (pos_preds[i] == neg_preds).mean()
             for i in range(m)]
        )
        V01 = np.array(
            [(neg_preds[j] < pos_preds).mean() + 0.5 * (neg_preds[j] == pos_preds).mean()
             for j in range(n)]
        )
        return V10, V01

    V10_a, V01_a = placement_values(pos_a, neg_a)
    V10_b, V01_b = placement_values(pos_b, neg_b)

    auc_a = V10_a.mean()
    auc_b = V10_b.mean()

    S10 = np.cov(np.vstack([V10_a, V10_b])) / m
    S01 = np.cov(np.vstack([V01_a, V01_b])) / n
    S   = S10 + S01

    L   = np.array([1.0, -1.0])
    var = L @ S @ L
    z   = (auc_a - auc_b) / np.sqrt(var)
    p   = 2.0 * (1.0 - stats.norm.cdf(abs(z)))
    return z, p, auc_a, auc_b


print('build_mlp and delong_auc_test defined.')

In [ ]:
test_inputs_wd = df_to_model_inputs(
    X_test, NUMERIC_FEATURES, CATEGORICAL_FEATURES, list(WIDE_CROSS_VOCAB_SIZES.keys())
)

best_wd_model = model1  # replace with best model from Part 3
wd_probs = best_wd_model.predict(test_inputs_wd, verbose=0).ravel()

X_dev_mlp  = X_dev.values.astype(np.float32)
X_test_mlp = X_test.values.astype(np.float32)

mlp_model = build_mlp(input_dim=X_dev_mlp.shape[1], hidden_layers=[128, 64])
mlp_model.fit(
    X_dev_mlp, y_dev,
    validation_split=0.15,
    epochs=50, batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=0)],
    verbose=1
)

mlp_probs = mlp_model.predict(X_test_mlp, verbose=0).ravel()

In [ ]:
fpr_wd,  tpr_wd,  _ = roc_curve(y_test, wd_probs)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_probs)

auc_wd  = auc(fpr_wd,  tpr_wd)
auc_mlp = auc(fpr_mlp, tpr_mlp)

plt.figure(figsize=(8, 6))
plt.plot(fpr_wd,  tpr_wd,  label=f'Wide & Deep (AUC = {auc_wd:.4f})')
plt.plot(fpr_mlp, tpr_mlp, label=f'MLP          (AUC = {auc_mlp:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: Wide & Deep vs. MLP')
plt.legend()
plt.tight_layout()
plt.show()

z, p, auc_a, auc_b = delong_auc_test(y_test, wd_probs, mlp_probs)
print(f'Wide & Deep AUC : {auc_a:.4f}')
print(f'MLP AUC         : {auc_b:.4f}')
print(f'DeLong z-stat   : {z:.4f}')
print(f'Two-sided p     : {p:.4f}')
if p < 0.05:
    print('=> Difference is statistically significant (α = 0.05).')
else:
    print('=> Difference is NOT statistically significant (α = 0.05).')

**Conclusion — Wide & Deep vs. MLP:**  
[Interpret the ROC curves, AUC values, and DeLong p-value. What does the wide branch contribute?]

---
## Part 5: Exceptional Credit — Embedding Visualization
*(Required for 7000-level student — 1pt)*

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


def visualize_embeddings(weights, category_labels, feature_name, method='pca'):
    if weights.shape[1] > 2:
        if method == 'tsne':
            perp = min(30, len(category_labels) - 1)
            reducer = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=perp)
        else:
            reducer = PCA(n_components=2)
        coords = reducer.fit_transform(weights)
    else:
        coords = weights

    plt.figure(figsize=(10, 8))
    plt.scatter(coords[:, 0], coords[:, 1], alpha=0.7)
    for i, label in enumerate(category_labels):
        plt.annotate(str(label), (coords[i, 0], coords[i, 1]), fontsize=8)
    plt.title(f'Embeddings for "{feature_name}" ({method.upper()})')
    plt.tight_layout()
    plt.show()


for col in CATEGORICAL_FEATURES:
    emb_layer = best_wd_model.get_layer(f'emb_{col}')
    weights   = emb_layer.get_weights()[0]
    print(f'{col}: embedding shape {weights.shape}')

    if col in label_encoders and hasattr(label_encoders[col], 'classes_'):
        cat_labels = label_encoders[col].classes_
    else:
        cat_labels = [str(i) for i in range(weights.shape[0])]

    visualize_embeddings(weights, cat_labels, col, method='pca')

**Embedding Analysis:**  
[For each plot explain what clusters are visible, whether semantically similar categories group
together, and what this reveals about what the model has learned.]

---
## Summary Table

| Model | Architecture | Val AUC (mean ± std) | Notes |
|-------|-------------|----------------------|-------|
| Wide & Deep 1 | deep=[128, 64] | — | Baseline |
| Wide & Deep 2 | deep=[256, 128, 64] | — | Higher dropout |
| Wide & Deep 3 | deep=[64, 32], emb=16 | — | Shallower |
| MLP | [128, 64] | — | No wide branch |

**Best model:** [Name and why]  
**Final test AUC (Wide & Deep):** [Value]  
**Final test AUC (MLP):** [Value]  
**DeLong p-value:** [Value]